In [ ]:
import kagglehub

path = kagglehub.dataset_download("mahdieizadpanah/birjand-university-mobile-palmprint-databasebmpd")

print("Path to dataset files:", path)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

try:
    print(f'Menggunakan path: {path}')
    for root, dirs, files in os.walk(path):
        if files:
            print(f'Direktori ditemukan: {root}')
            print(f'Contoh file: {files[:5]}')
            break
except NameError:
    print("Error: Variabel 'path' belum didefinisikan. Silakan jalankan sel pertama (download dataset) terlebih dahulu.")

In [ ]:
def preprocess_palmprint(image_path, use_median=False):
    # Load image dalam grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Resize untuk standarisasi (opsional, misal 256x256)
    img = cv2.resize(img, (256, 256))

    if use_median:
        # Median Filter - efektif menghilangkan salt-and-pepper noise, menjaga tepi
        filtered = cv2.medianBlur(img, 5)
    else:
        # Gaussian Blur untuk mengurangi noise sebelum adaptive thresholding
        filtered = cv2.GaussianBlur(img, (5, 5), 0)

    # Adaptive Thresholding
    thresh = cv2.adaptiveThreshold(
        filtered, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )

    return img, thresh

example_img_path = None
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith(('.bmp', '.jpg', '.png')):
            example_img_path = os.path.join(root, f)
            break
    if example_img_path: break

if example_img_path:
    original_gaussian, processed_gaussian = preprocess_palmprint(example_img_path, use_median=False)
    original_median, processed_median = preprocess_palmprint(example_img_path, use_median=True)

    # Simpan hasil preprocessing (Gaussian Blur + Adaptive Threshold) untuk tahap selanjutnya
    original = original_gaussian
    preprocessed = processed_gaussian

    # Visualisasi perbandingan kedua metode preprocessing
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.title('Original Gray')
    plt.imshow(original, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title('Adaptive Thresholding\n(Gaussian Blur pre-filter)')
    plt.imshow(processed_gaussian, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title('Adaptive Thresholding\n(Median Filter pre-filter)')
    plt.imshow(processed_median, cmap='gray')
    plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('Tidak ditemukan file gambar dalam dataset.')


In [ ]:
def apply_histogram_equalization(img):
    """Menerima gambar numpy array (hasil preprocessing) sebagai input."""
    if img is None:
        return None

    # 1. Global Histogram Equalization
    equ = cv2.equalizeHist(img)

    # 2. CLAHE (Contrast Limited Adaptive Histogram Equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl1 = clahe.apply(img)

    return img, equ, cl1

if example_img_path and 'preprocessed' in locals():
    # Gunakan hasil preprocessing sebagai input intensity transformation
    original_in, global_eq, clahe_eq = apply_histogram_equalization(preprocessed)

    # Simpan hasil intensity transformation untuk tahap spatial filtering
    intensity_transformed = clahe_eq

    # Visualisasi perbandingan
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.title('Preprocessed (Input)')
    plt.imshow(original_in, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title('Global Hist Equalization')
    plt.imshow(global_eq, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title('CLAHE (Adaptive)')
    plt.imshow(clahe_eq, cmap='gray')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    # Plot Histogram untuk melihat distribusi intensitas
    plt.figure(figsize=(15, 4))
    plt.subplot(1, 3, 1)
    plt.hist(original_in.ravel(), bins=256, range=(0, 256))
    plt.title('Hist: Preprocessed Input')

    plt.subplot(1, 3, 2)
    plt.hist(global_eq.ravel(), bins=256, range=(0, 256))
    plt.title('Hist: Global EQ')

    plt.subplot(1, 3, 3)
    plt.hist(clahe_eq.ravel(), bins=256, range=(0, 256))
    plt.title('Hist: CLAHE')
    plt.show()
else:
    print('Hasil preprocessing tidak ditemukan. Jalankan sel preprocessing terlebih dahulu.')


In [ ]:
def apply_gamma_correction(img, gamma=1.0):
    """Menerima gambar numpy array (hasil preprocessing) sebagai input."""
    if img is None:
        return None

    # Membangun lookup table untuk gamma correction
    # Rumus: s = c * r^gamma
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255
                      for i in np.arange(0, 256)]).astype("uint8")

    # Mengaplikasikan transformasi menggunakan lookup table (lebih cepat)
    return cv2.LUT(img, table)

if example_img_path and 'preprocessed' in locals():
    # Gunakan hasil preprocessing sebagai input intensity transformation
    gamma_05 = apply_gamma_correction(preprocessed, gamma=0.5)  # Lebih gelap
    gamma_15 = apply_gamma_correction(preprocessed, gamma=1.5)  # Lebih terang
    gamma_22 = apply_gamma_correction(preprocessed, gamma=2.2)  # Sangat terang

    # Visualisasi perbandingan
    plt.figure(figsize=(20, 5))

    plt.subplot(1, 4, 1)
    plt.title('Preprocessed Input\n(Gamma 1.0)')
    plt.imshow(preprocessed, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.title('Gamma 0.5 (Darker)')
    plt.imshow(gamma_05, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.title('Gamma 1.5 (Brighter)')
    plt.imshow(gamma_15, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.title('Gamma 2.2 (Very Bright)')
    plt.imshow(gamma_22, cmap='gray')
    plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('Hasil preprocessing tidak ditemukan. Jalankan sel preprocessing terlebih dahulu.')


In [ ]:
import os

# Collecting all pictures path into a list
all_images = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith(('.bmp', '.jpg', '.png')):
            all_images.append(os.path.join(root, f))

print(f"Total gambar yang tersedia: {len(all_images)}")

# Memasukkan index foto
pilih_index = 10

if 0 <= pilih_index < len(all_images):
    selected_path = all_images[pilih_index]
    print(f"Gambar terpilih (Index {pilih_index}): {os.path.basename(selected_path)}")

    # 1. PREPROCESSING
    orig, thresh = preprocess_palmprint(selected_path, use_median=False)

    # 2. INTENSITY TRANSFORMATION (menggunakan hasil preprocessing)
    _, _, clahe_result = apply_histogram_equalization(thresh)
    gamma_result = apply_gamma_correction(thresh, gamma=1.5)

    # Visualisasi alur: Preprocessing -> Intensity Transformation
    plt.figure(figsize=(20, 5))

    plt.subplot(1, 4, 1)
    plt.title('1. Original')
    plt.imshow(orig, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.title('2. Preprocessing\n(Adaptive Threshold)')
    plt.imshow(thresh, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.title('3. Intensity Transformation\n(CLAHE pada Preprocessed)')
    plt.imshow(clahe_result, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.title('3. Intensity Transformation\n(Gamma 1.5 pada Preprocessed)')
    plt.imshow(gamma_result, cmap='gray')
    plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print(f"Index {pilih_index} di luar jangkauan! Pilih antara 0 sampai {len(all_images)-1}")


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def apply_spatial_filtering(img):
    """Menerima gambar numpy array (hasil intensity transformation) sebagai input."""
    # --- 1. SMOOTHING FILTERS (Low-pass Filters) ---

    # Averaging Filter (Mean Filter) - Menggunakan kernel 5x5
    blur_avg = cv2.blur(img, (5, 5))

    # Median Filter - Sangat efektif untuk menghilangkan noise bintik (salt-and-pepper)
    blur_median = cv2.medianBlur(img, 5)

    # Gaussian Filter - Memperhalus citra dengan mempertahankan struktur lebih baik dari averaging
    blur_gaussian = cv2.GaussianBlur(img, (5, 5), 0)

    # --- 2. SHARPENING FILTERS (High-pass Filters) ---

    # Laplacian Filter - Menekankan perubahan intensitas yang cepat (tepi)
    laplacian = cv2.Laplacian(img, cv2.CV_64F)
    laplacian_abs = cv2.convertScaleAbs(laplacian)

    # Sobel Filter - Bagus untuk deteksi tepi horizontal dan vertikal
    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)  # Horizontal
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)  # Vertical
    sobel_combined = cv2.magnitude(sobelx, sobely)
    sobel_final = cv2.convertScaleAbs(sobel_combined)

    # Canny Edge Detection
    canny = cv2.Canny(img, 50, 150)

    # Unsharp Masking - Teknik untuk meningkatkan ketajaman detail
    gaussian_for_unsharp = cv2.GaussianBlur(img, (5, 5), 0)
    unsharp_mask = cv2.addWeighted(img, 1.5, gaussian_for_unsharp, -0.5, 0)

    return {
        "Original": img,
        "Averaging": blur_avg,
        "Median": blur_median,
        "Gaussian": blur_gaussian,
        "Laplacian": laplacian_abs,
        "Sobel": sobel_final,
        "Canny": canny,
        "Unsharp Masking": unsharp_mask
    }

# Gunakan hasil intensity transformation sebagai input spatial filtering
if 'intensity_transformed' in locals():
    results = apply_spatial_filtering(intensity_transformed)

    # Visualisasi Hasil Smoothing
    plt.figure(figsize=(20, 10))
    plt.suptitle('Spatial Filtering pada Hasil Intensity Transformation', fontsize=14)

    titles_smooth = ["Original", "Averaging", "Median", "Gaussian"]
    for i, title in enumerate(titles_smooth):
        plt.subplot(2, 4, i + 1)
        plt.title(title)
        plt.imshow(results[title], cmap='gray')
        plt.axis('off')

    # Visualisasi Hasil Sharpening / Edge Detection
    titles_sharp = ["Laplacian", "Sobel", "Canny", "Unsharp Masking"]
    for i, title in enumerate(titles_sharp):
        plt.subplot(2, 4, i + 5)
        plt.title(title)
        plt.imshow(results[title], cmap='gray')
        plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("Variabel 'intensity_transformed' tidak ditemukan. Jalankan sel intensity transformation (Cell 3) terlebih dahulu.")


In [ ]:
def complete_pcd_pipeline(image_path, use_median=False, gamma=1.5):
    """
    Pipeline lengkap PCD:
    Preprocessing -> Intensity Transformation -> Spatial Filtering

    Args:
        image_path: Path ke file gambar
        use_median: Jika True, gunakan Median Filter; jika False, gunakan Gaussian Blur sebelum adaptive thresholding
        gamma: Nilai gamma untuk gamma correction
    """
    # === TAHAP 1: PREPROCESSING ===
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print("Gagal membaca gambar!")
        return
    img = cv2.resize(img, (256, 256))

    original, preprocessed = preprocess_palmprint(image_path, use_median=use_median)
    method_label = 'Median Filter' if use_median else 'Gaussian Blur'

    # === TAHAP 2: INTENSITY TRANSFORMATION (input: preprocessed) ===
    _, _, clahe_result = apply_histogram_equalization(preprocessed)
    gamma_result = apply_gamma_correction(preprocessed, gamma=gamma)

    # Pilih hasil intensity transformation untuk dilanjutkan ke spatial filtering
    intensity_out = clahe_result

    # === TAHAP 3: SPATIAL FILTERING (input: intensity_transformed) ===
    spatial_results = apply_spatial_filtering(intensity_out)

    # === VISUALISASI PIPELINE LENGKAP ===
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    fig.suptitle(f'Pipeline PCD Lengkap (Preprocessing: {method_label})', fontsize=16)

    # Baris 1: Preprocessing
    axes[0, 0].imshow(original, cmap='gray'); axes[0, 0].set_title('1a. Original Gray'); axes[0, 0].axis('off')
    axes[0, 1].imshow(preprocessed, cmap='gray'); axes[0, 1].set_title(f'1b. Preprocessing\n(Adaptive Thresh + {method_label})'); axes[0, 1].axis('off')
    axes[0, 2].axis('off')
    axes[0, 3].axis('off')

    # Baris 2: Intensity Transformation
    axes[1, 0].imshow(preprocessed, cmap='gray'); axes[1, 0].set_title('2. Input (Preprocessed)'); axes[1, 0].axis('off')
    axes[1, 1].imshow(clahe_result, cmap='gray'); axes[1, 1].set_title('2a. CLAHE'); axes[1, 1].axis('off')
    axes[1, 2].imshow(gamma_result, cmap='gray'); axes[1, 2].set_title(f'2b. Gamma Correction ({gamma})'); axes[1, 2].axis('off')
    axes[1, 3].axis('off')

    # Baris 3: Spatial Filtering
    axes[2, 0].imshow(intensity_out, cmap='gray'); axes[2, 0].set_title('3. Input (CLAHE)'); axes[2, 0].axis('off')
    axes[2, 1].imshow(spatial_results['Sobel'], cmap='gray'); axes[2, 1].set_title('3a. Sobel Edge Detection'); axes[2, 1].axis('off')
    axes[2, 2].imshow(spatial_results['Canny'], cmap='gray'); axes[2, 2].set_title('3b. Canny Edge Detection'); axes[2, 2].axis('off')
    axes[2, 3].imshow(spatial_results['Unsharp Masking'], cmap='gray'); axes[2, 3].set_title('3c. Unsharp Masking\n(Sharpening)'); axes[2, 3].axis('off')

    plt.tight_layout()
    plt.show()

    return {
        'original': original,
        'preprocessed': preprocessed,
        'clahe': clahe_result,
        'gamma': gamma_result,
        'spatial': spatial_results
    }


# Eksekusi pipeline lengkap
if example_img_path:
    print("=== Pipeline dengan Gaussian Blur + Adaptive Thresholding ===")
    pipeline_result = complete_pcd_pipeline(example_img_path, use_median=False, gamma=1.5)

    print("\n=== Pipeline dengan Median Filter + Adaptive Thresholding ===")
    pipeline_result_median = complete_pcd_pipeline(example_img_path, use_median=True, gamma=1.5)
else:
    print('Contoh gambar tidak ditemukan.')
